In [4]:
! pip install transformer_lens

  Using cached transformer_lens-2.16.1-py3-none-any.whl.metadata (12 kB)
  Using cached beartype-0.14.1-py3-none-any.whl.metadata (28 kB)
  Using cached better_abc-0.0.3-py3-none-any.whl.metadata (1.4 kB)
  Using cached fancy_einsum-0.0.3-py3-none-any.whl.metadata (1.2 kB)
  Using cached jaxtyping-0.3.4-py3-none-any.whl.metadata (7.8 kB)
  Using cached numpy-1.26.4-cp312-cp312-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (61 kB)
  Using cached transformers_stream_generator-0.0.5-py3-none-any.whl
  Using cached wadler_lindig-0.1.7-py3-none-any.whl.metadata (17 kB)
Using cached transformer_lens-2.16.1-py3-none-any.whl (192 kB)
Using cached beartype-0.14.1-py3-none-any.whl (739 kB)
Using cached better_abc-0.0.3-py3-none-any.whl (3.5 kB)
Using cached fancy_einsum-0.0.3-py3-none-any.whl (6.2 kB)
Using cached jaxtyping-0.3.4-py3-none-any.whl (56 kB)
Using cached numpy-1.26.4-cp312-cp312-manylinux_2_17_x86_64.manylinux2014_x86_64.whl (18.0 MB)
Using cached wadler_lindig-0.1.7-py3-n

In [1]:
import os
import torch
import matplotlib.pyplot as plt
from transformer_lens import HookedTransformer


Reverrse activation patching

In [6]:
class ReverseActivationPatcher:
    """Run reverse activation patching to measure uncertainty mass shifts."""

    def __init__(
        self,
        model_name,
        prompt_real,
        prompt_fake,
        results_dir="./results_reverse_activation_patching",
        patch_type="resid",
        patch_targets=None,
    ):
        self.model_name = model_name
        self.prompt_real = prompt_real
        self.prompt_fake = prompt_fake
        self.results_dir = results_dir
        self.patch_type = patch_type  # "resid" or "head"
        self.patch_targets = patch_targets
        self.device = "cuda" if torch.cuda.is_available() else "cpu"
        self.dtype = torch.float16 if torch.cuda.is_available() else torch.float32

        os.makedirs(self.results_dir, exist_ok=True)
        self.out_dir = os.path.join(self.results_dir, model_name.replace("/", "_"))
        os.makedirs(self.out_dir, exist_ok=True)

        print(f"Loading {model_name} on {self.device} ({self.dtype})...")
        self.model = HookedTransformer.from_pretrained(model_name, device=self.device, dtype=self.dtype)
        self.uncertainty_tokens = self._get_uncertainty_tokens()
        print(f"Uncertainty vocab size: {len(self.uncertainty_tokens)}")

    def _get_uncertainty_tokens(self):
        words = [
            "unknown", "unsure", "don", "'t", "know", "can't", "unable", "impossible",
            "doesn't", "exist", "fictional", "imaginary", "no", "such", "thing",
            "hypothetical", "made", "up", "never", "happened",
        ]
        token_ids = []
        for w in words:
            try:
                toks = self.model.to_tokens(" " + w, prepend_bos=False)[0]
                token_ids.extend(toks.tolist())
            except Exception:
                continue
        return sorted(set(token_ids))

    @staticmethod
    def _uncertainty_mass(logits, token_ids):
        probs = torch.softmax(logits, dim=-1)
        if not isinstance(token_ids, list) and not isinstance(token_ids, torch.Tensor):
            token_ids = list(token_ids)
        return probs[token_ids].sum().item()

    def _collect_cache(self, prompt):
        tokens = self.model.to_tokens(prompt)
        if self.patch_type == "resid":
            names_filter = lambda n: "hook_resid_post" in n
        else:
            names_filter = lambda n: "attn.hook_z" in n
        with torch.no_grad():
            logits, cache = self.model.run_with_cache(tokens, names_filter=names_filter)
        return logits[0, -1], cache

    def _patch_layer(self, base_tokens, fake_cache, layer_idx):
        hook_name = f"blocks.{layer_idx}.hook_resid_post"
        fake_resid = fake_cache[hook_name]

        def patch_fn(value, hook):
            # patch the last token's residual stream with the value from the fake cache
            value = value.clone()
            value[:, -1, :] = fake_resid[:, -1, :]
            return value

        with torch.no_grad():
            logits = self.model.run_with_hooks(
                base_tokens,
                fwd_hooks=[(hook_name, patch_fn)],
            )
        return logits[0, -1]

    def _patch_head(self, base_tokens, fake_cache, layer_idx, head_idx):
        hook_name = f"blocks.{layer_idx}.attn.hook_z"
        fake_z = fake_cache[hook_name]

        def patch_fn(value, hook):
            # patch the last token's attention head output with the value from the fake cache
            value = value.clone()
            value[:, -1, head_idx, :] = fake_z[:, -1, head_idx, :]
            return value

        with torch.no_grad():
            logits = self.model.run_with_hooks(
                base_tokens,
                fwd_hooks=[(hook_name, patch_fn)],
            )
        return logits[0, -1]

    def run(self):
        logits_real, self.cache_real = self._collect_cache(self.prompt_real)
        # We need self.cache_fake for generate_with_patch
        _, self.cache_fake = self._collect_cache(self.prompt_fake)

        mass_real = self._uncertainty_mass(logits_real, self.uncertainty_tokens)
        print(f"Baseline (real):  {mass_real:.4e}")

        patched_mass = []
        pct_change = []

        base_tokens = self.model.to_tokens(self.prompt_real)

        if self.patch_type == "resid":
            layers = list(range(self.model.cfg.n_layers)) if self.patch_targets is None else self.patch_targets
            labels = [f"L{l}" for l in layers]
            for layer in layers:
                logits_patched = self._patch_layer(base_tokens, self.cache_fake, layer)
                mass_patched = self._uncertainty_mass(logits_patched, self.uncertainty_tokens)
                patched_mass.append(mass_patched)
                delta = mass_patched - mass_real
                pct = (delta / mass_real * 100) if mass_real != 0 else 0.0
                pct_change.append(pct)
                print(f"Layer {layer:02d}: patched mass={mass_patched:.4e}, pct change={pct:+.2f}%")
        else:
            if not self.patch_targets:
                raise ValueError("For head patching, provide patch_targets as list of (layer, head)")
            layers_heads = self.patch_targets
            labels = [f"L{l}H{h}" for l, h in layers_heads]
            for (layer, head) in layers_heads:
                logits_patched = self._patch_head(base_tokens, self.cache_fake, layer, head)
                mass_patched = self._uncertainty_mass(logits_patched, self.uncertainty_tokens)
                patched_mass.append(mass_patched)
                delta = mass_patched - mass_real
                pct = (delta / mass_real * 100) if mass_real != 0 else 0.0
                pct_change.append(pct)
                print(f"L{layer}H{head}: patched mass={mass_patched:.4e}, pct change={pct:+.2f}%")

        self._plot(labels, patched_mass, pct_change, mass_real)

        return {
            "mass_real": mass_real,
            "patched_mass": patched_mass,
            "pct_change": pct_change,
            "out_dir": self.out_dir,
        }

    def _plot(self, labels, patched_mass, pct_change, mass_real):
        x = list(range(len(labels)))
        fig, axes = plt.subplots(2, 1, figsize=(10, 8), sharex=True)
        fig.suptitle(f"Reverse Activation Patching: {self.model_name}", fontweight="bold")

        axes[0].plot(x, [mass_real] * len(x), label="Baseline (real)", color="red", linestyle="--")
        axes[0].plot(x, patched_mass, label="Patched (fake)", color="blue")
        axes[0].set_ylabel("Uncertainty mass")
        axes[0].grid(alpha=0.3)
        axes[0].legend()

        axes[1].axhline(0, color="black", linewidth=1)
        axes[1].bar(x, pct_change, color=["green" if p < 0 else "orange" for p in pct_change])
        axes[1].set_xticks(x)
        axes[1].set_xticklabels(labels, rotation=45, ha="right")
        axes[1].set_xlabel("Patch target")
        axes[1].set_ylabel("% change vs real")
        axes[1].grid(alpha=0.3)

        plt.tight_layout(rect=[0, 0, 1, 0.95])
        out_path = os.path.join(self.out_dir, "reverse_activation_patching.png")
        plt.savefig(out_path, dpi=150)
        plt.close()
        print(f"Saved plot: {out_path}")

    def generate_with_patch(self, prompt=None, max_new_tokens=20, temperature=0.7):
        """Generate text while applying the configured patch hooks.

        Uses the fake-cache from the counterfactual prompt to patch either
        residual streams (layer list) or specific heads (layer, head) onto
        the real prompt.
        """
        if not hasattr(self, "cache_fake"):
            raise RuntimeError("Run patcher.run() first to collect fake cache")

        prompt_to_generate = prompt or self.prompt_real # base generation on the real prompt
        tokens = self.model.to_tokens(prompt_to_generate)
        hooks = []

        if self.patch_type == "resid":
            layers = list(range(self.model.cfg.n_layers)) if self.patch_targets is None else self.patch_targets
            for layer in layers:
                hook_name = f"blocks.{layer}.hook_resid_post"
                fake_resid = self.cache_fake[hook_name]

                def make_resid_hook(fake_resid_val):
                    def hook_fn(value, hook):
                        value = value.clone()
                        value[:, -1, :] = fake_resid_val[:, -1, :]
                        return value
                    return hook_fn

                hooks.append((hook_name, make_resid_hook(fake_resid)))
        else:
            if not self.patch_targets:
                raise ValueError("For head patching, provide patch_targets as list of (layer, head)")
            for layer, head in self.patch_targets:
                hook_name = f"blocks.{layer}.attn.hook_z"
                fake_z = self.cache_fake[hook_name]

                def make_head_hook(fake_z_val, head_idx):
                    def hook_fn(value, hook):
                        value = value.clone()
                        value[:, -1, head_idx, :] = fake_z_val[:, -1, head_idx, :]
                        return value
                    return hook_fn

                hooks.append((hook_name, make_head_hook(fake_z, head)))

        with torch.no_grad():
            generated = self.model.generate(
                tokens,
                max_new_tokens=max_new_tokens,
                temperature=temperature,
                fwd_hooks=hooks,
            )

        text = self.model.tokenizer.decode(generated[0].tolist())
        return text, generated

print("ReverseActivationPatcher class defined.")

ReverseActivationPatcher class defined.


In [7]:
MODEL_NAME = "meta-llama/Llama-3.2-1B-Instruct"
PROMPT_REAL = "What is the freezing point of water at standard atmospheric pressure?"
PROMPT_FAKE = "What is the freezing point of water on Planet Xylon, where gravity is twice that of Earth?"

patch_targets = [(11, 0), (11, 3)]  # list of (layer, head)

reverse_patcher = ReverseActivationPatcher(
    MODEL_NAME,
    PROMPT_REAL,
    PROMPT_FAKE,
    patch_type="head",
    patch_targets=patch_targets,
)

results = reverse_patcher.run()
print("Reverse activation patching results:")
print(results)

Loading meta-llama/Llama-3.2-1B-Instruct on cuda (torch.float16)...


Loaded pretrained model meta-llama/Llama-3.2-1B-Instruct into HookedTransformer
Uncertainty vocab size: 22
Baseline (real):  8.8751e-05
L11H0: patched mass=8.7976e-05, pct change=-0.87%
L11H3: patched mass=9.1195e-05, pct change=+2.75%
Saved plot: ./results_reverse_activation_patching/meta-llama_Llama-3.2-1B-Instruct/reverse_activation_patching.png
Reverse activation patching results:
{'mass_real': 8.875131607055664e-05, 'patched_mass': [8.797645568847656e-05, 9.119510650634766e-05], 'pct_change': [-0.8730691739422431, 2.7535258562793823], 'out_dir': './results_reverse_activation_patching/meta-llama_Llama-3.2-1B-Instruct'}


In [8]:
MODEL_NAME = "qwen2.5-1.5b-instruct"
PROMPT_REAL = "What is the freezing point of water at standard atmospheric pressure?"
PROMPT_FAKE = "What is the freezing point of water on Planet Xylon, where gravity is twice that of Earth?"

patch_targets = [ (8, 4), (8, 8), (8, 11)]   # list of (layer, head)

reverse_patcher = ReverseActivationPatcher(
    MODEL_NAME,
    PROMPT_REAL,
    PROMPT_FAKE,
    patch_type="head",
    patch_targets=patch_targets,
)

results = reverse_patcher.run()
print("Reverse activation patching results:")
print(results)

Loading qwen2.5-1.5b-instruct on cuda (torch.float16)...


Loaded pretrained model qwen2.5-1.5b-instruct into HookedTransformer
Uncertainty vocab size: 22
Baseline (real):  4.8280e-06
L8H4: patched mass=4.8280e-06, pct change=+0.00%
L8H8: patched mass=4.7088e-06, pct change=-2.47%
L8H11: patched mass=5.0068e-06, pct change=+3.70%
Saved plot: ./results_reverse_activation_patching/qwen2.5-1.5b-instruct/reverse_activation_patching.png
Reverse activation patching results:
{'mass_real': 4.827976226806641e-06, 'patched_mass': [4.827976226806641e-06, 4.708766937255859e-06, 5.0067901611328125e-06], 'pct_change': [0.0, -2.4691358024691357, 3.7037037037037033], 'out_dir': './results_reverse_activation_patching/qwen2.5-1.5b-instruct'}


In [9]:
!zip -r /content/results_reverse_activation_patching.zip /content/results_reverse_activation_patching/

  adding: content/results_reverse_activation_patching/ (stored 0%)
  adding: content/results_reverse_activation_patching/meta-llama_Llama-3.2-1B-Instruct/ (stored 0%)
  adding: content/results_reverse_activation_patching/meta-llama_Llama-3.2-1B-Instruct/reverse_activation_patching.png (deflated 18%)
  adding: content/results_reverse_activation_patching/qwen2.5-1.5b-instruct/ (stored 0%)
  adding: content/results_reverse_activation_patching/qwen2.5-1.5b-instruct/reverse_activation_patching.png (deflated 17%)
